# Virtual-pipeline — Commercial VTuber Builder

이 저장소는 **private**입니다. Colab의 `/github/...` 로더는 private GitHub Contents API를 인증 없이 읽을 수 없으므로 README의 직접 Colab 링크를 사용하지 않습니다.

이 노트북을 Colab에 업로드한 뒤 첫 Setup 셀을 실행하면:
1. Colab Secret `GITHUB_TOKEN`(없으면 1회 비밀 입력)으로 private 저장소 인증
2. `Virtual-pipeline`의 **최신 `origin/main`을 clone/fetch + hard reset**
3. pinned TripoSR commit checkout
4. 최신 저장소를 editable install
5. 이후 Avatar/Accessory 단계는 방금 받은 최신 main 코드만 사용

- Avatar Mode: 이미지 → TripoSR → canonical fitting → rig/expressions/gaze/SpringBone → VRM 1.0
- Accessory Mode: 완성 VRM + 악세사리 이미지들 → 3D → anchor fitting/collision → static bake

GitHub 토큰은 코드/출력에 저장하지 않습니다. Fine-grained PAT라면 이 private repo에 **Contents: Read-only** 권한이면 됩니다.


## 1. Setup

In [ ]:
import getpass, os, pathlib, subprocess, sys

if sys.version_info < (3, 12):
    raise RuntimeError(f'Python 3.12+ required, current={sys.version}')

def run(cmd, *, cwd=None, env=None):
    print('+', ' '.join(str(x) for x in cmd if 'token' not in str(x).lower()))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

# ---- GitHub private-repo authentication ----
try:
    from google.colab import userdata
    try:
        github_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        github_token = None
except Exception:
    github_token = None

if not github_token:
    github_token = getpass.getpass('GitHub token (private repo Contents: Read): ').strip()
if not github_token:
    raise RuntimeError('GITHUB_TOKEN is required because Virtual-pipeline is private.')

askpass = pathlib.Path('/tmp/virtual-pipeline-git-askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\nesac\n')
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    'GITHUB_TOKEN': github_token,
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
})

# ---- Always synchronize the latest Virtual-pipeline main ----
repo = pathlib.Path('/content/Virtual-pipeline')
repo_url = 'https://github.com/jujumelona/Virtual-pipeline.git'
if (repo / '.git').is_dir():
    run(['git', '-C', str(repo), 'remote', 'set-url', 'origin', repo_url], env=git_env)
    run(['git', '-C', str(repo), 'fetch', '--prune', 'origin', 'main'], env=git_env)
else:
    if repo.exists():
        import shutil
        shutil.rmtree(repo)
    run(['git', 'clone', '--branch', 'main', '--single-branch', repo_url, str(repo)], env=git_env)

run(['git', '-C', str(repo), 'checkout', '-B', 'main', 'origin/main'], env=git_env)
run(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], env=git_env)
head = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
remote_head = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'origin/main'], text=True).strip()
if head != remote_head:
    raise RuntimeError(f'Local main is not current: local={head}, origin/main={remote_head}')
print('Virtual-pipeline main:', head)

# Token is no longer needed after repository synchronization.
git_env.pop('GITHUB_TOKEN', None)
github_token = None
try:
    askpass.unlink()
except FileNotFoundError:
    pass

# ---- Pinned TripoSR ----
TRIPOSR_COMMIT = '107cefdc244c39106fa830359024f6a2f1c78871'
triposr = pathlib.Path('/content/third_party/TripoSR')
triposr.parent.mkdir(parents=True, exist_ok=True)
if (triposr / '.git').is_dir():
    run(['git', '-C', str(triposr), 'fetch', '--prune', 'origin'])
else:
    if triposr.exists():
        import shutil
        shutil.rmtree(triposr)
    run(['git', 'clone', 'https://github.com/VAST-AI-Research/TripoSR.git', str(triposr)])
run(['git', '-C', str(triposr), 'checkout', '--detach', TRIPOSR_COMMIT])
actual_triposr = subprocess.check_output(['git', '-C', str(triposr), 'rev-parse', 'HEAD'], text=True).strip()
if actual_triposr != TRIPOSR_COMMIT:
    raise RuntimeError(f'TripoSR pin mismatch: {actual_triposr}')

# ---- Install exactly the freshly synchronized repository ----
run([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'])
run([sys.executable, '-m', 'pip', 'install', '-r', str(triposr / 'requirements.txt')])
run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)])

os.environ['TRIPOSR_DIR'] = str(triposr)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

# Remove any stale modules imported before the sync/install.
for module_name in list(sys.modules):
    if module_name == 'vtuber_pipeline' or module_name.startswith('vtuber_pipeline.'):
        del sys.modules[module_name]

print('Setup complete')
print('  repo:', repo)
print('  main:', head)
print('  TripoSR:', actual_triposr)


## 2. Avatar Mode

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Avatar Mode에서는 캐릭터 이미지 1장만 업로드하세요.')

name, data = next(iter(uploaded.items()))
avatar_input_dir = Path('/content/input/avatar')
avatar_input_dir.mkdir(parents=True, exist_ok=True)
avatar_image = avatar_input_dir / name
avatar_image.write_bytes(data)
print('Avatar input:', avatar_image)

In [ ]:
import json
from vtuber_pipeline.avatar import build_avatar

avatar_output = '/content/output/avatar'
avatar_result = build_avatar(
    image_path=str(avatar_image),
    output_dir=avatar_output,
    config={
        'profile': 'commercial',
        'commercial_usage': 'corporation',
    },
)

print(json.dumps({
    'status': avatar_result.get('status'),
    'failed_stages': avatar_result.get('failed_stages'),
    'failed_reason': avatar_result.get('failed_reason'),
    'vrm_path': avatar_result.get('vrm_path'),
}, ensure_ascii=False, indent=2))

if avatar_result.get('status') != 'complete':
    for stage, result in avatar_result.get('stages', {}).items():
        print(stage, '=>', result.get('status'), result.get('error') or result.get('warning'))
    raise RuntimeError('Avatar build did not complete.')

avatar_vrm_path = avatar_result['vrm_path']
print('Avatar VRM:', avatar_vrm_path)

In [ ]:
from google.colab import files
files.download(avatar_vrm_path)

## 3. Accessory Mode

Avatar Mode가 끝난 뒤 별도로 실행합니다. 여러 이미지를 한 번에 업로드할 수 있습니다.

ANCHORS를 업로드 순서에 맞게 수정할 수 있습니다.
지원 예: HEAD_TOP, FACE, LEFT_EAR, RIGHT_EAR, NECK, CHEST, BACK, LEFT_HAND, RIGHT_HAND, HIPS.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded_accessories = files.upload()
acc_input_dir = Path('/content/input/accessories')
acc_input_dir.mkdir(parents=True, exist_ok=True)

accessory_images = []
for name, data in uploaded_accessories.items():
    p = acc_input_dir / name
    p.write_bytes(data)
    accessory_images.append(str(p))

ANCHORS = ['HEAD_TOP'] * len(accessory_images)
print(list(zip(accessory_images, ANCHORS)))

In [ ]:
from vtuber_pipeline.accessory import reconstruct_accessories, AccessoryPipeline

recon_dir = '/content/output/accessory_reconstruction'
reconstructed = reconstruct_accessories(accessory_images, recon_dir)

current_vrm = avatar_vrm_path
for idx, (item, anchor) in enumerate(zip(reconstructed, ANCHORS)):
    if item.get('status') != 'complete':
        raise RuntimeError(f'Accessory reconstruction failed: {item}')

    out_dir = f'/content/output/accessory_{idx:03d}'
    result = AccessoryPipeline(out_dir).build(
        base_vrm=current_vrm,
        accessory_glb=item['mesh'],
        config={
            'anchor_name': anchor,
            'physics': {'enabled': False},
        },
    )
    print(idx, anchor, result.get('status'))
    if result.get('status') != 'complete':
        raise RuntimeError(f'Accessory pipeline failed: {result}')
    current_vrm = result['output_vrm']

combined_vrm_path = current_vrm
print('Combined VRM:', combined_vrm_path)

In [ ]:
from google.colab import files
files.download(combined_vrm_path)